In [1]:
# 202431052_Anfal Ridho Hanif
!pip install requests beautifulsoup4 pandas

In [2]:
# 202431052_Anfal Ridho Hanif
import csv
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib import robotparser

BASE = "https://komikcast.info"
LIST_URL = BASE + "/manga?sort=latest_update&page={}"
HEADERS = {"User-Agent": "Mozilla/5.0 (Tugas Praktikum Scraping IT PLN)"}

MIN_DATA = 200   
TARGET = 260    
MAX_PAGE = 30    
DELAY = 2       

In [3]:
# 202431052_Anfal Ridho Hanif
resp_robots = requests.get(BASE + "/robots.txt", headers=HEADERS, timeout=20)
print("Status robots.txt:", resp_robots.status_code)
print(resp_robots.text)

rp = robotparser.RobotFileParser()
rp.parse(resp_robots.text.splitlines())
for p in (1, 2):
    url = LIST_URL.format(p)
    print("Boleh diambil?", url, "->", rp.can_fetch("*", url))

Status robots.txt: 200
User-agent: *
Allow: /
Disallow: /admin
Disallow: /admin/*
Disallow: /login
Disallow: /register
Disallow: /api/

Sitemap: https://komikcast.info/sitemap.xml

Boleh diambil? https://komikcast.info/manga?sort=latest_update&page=1 -> True
Boleh diambil? https://komikcast.info/manga?sort=latest_update&page=2 -> True


In [4]:
# 202431052_Anfal Ridho Hanif
def cari_blok(soup):
    """Cari semua blok data (satu blok = satu kartu komik).

    Tiap kartu punya satu link judul ke /manga/<slug>. Dari link itu kita
    naik ke elemen induk terdekat yang memuat teks 'Rating' = blok kartu.
    """
    blok, sudah = [], set()
    for a in soup.find_all("a", href=re.compile(r"/manga/[^/?#]+$")):
        if not a.get_text(strip=True):      # lewati link gambar cover
            continue
        node = a
        for _ in range(6):
            node = node.parent
            if node is None:
                break
            teks = node.get_text(" ", strip=True)
            if "Rating" in teks and "Status" in teks:
                break
        else:
            node = None
        if node is not None and id(node) not in sudah:
            sudah.add(id(node))
            blok.append(node)
    return blok


def ambil_data(b):
    """Ambil tiap atribut dari satu blok (hasilnya masih teks)."""
    judul = b.find("a", href=re.compile(r"/manga/[^/?#]+$"), string=True)
    judul = judul.get_text(strip=True) if judul else ""

    # chapter terbaru: angka diambil dari href link chapter (/chapter/56)
    nomor = []
    for c in b.find_all("a", href=re.compile(r"/chapter/")):
        akhir = c.get("href", "").rstrip("/").rsplit("/", 1)[-1]
        if re.fullmatch(r"\d+(\.\d+)?", akhir):
            nomor.append(float(akhir))
    chapter = str(max(nomor)) if nomor else ""

    teks = b.get_text(" ", strip=True)
    status = re.findall(r"Status\s*([A-Za-z]+)", teks)
    tipe = re.findall(r"Type\s*([A-Za-z]+)", teks)
    rating = re.findall(r"Rating\s*★?\s*([\d.]+)", teks)

    return {
        "judul": judul,
        "tipe": tipe[-1] if tipe else "",
        "status": status[-1] if status else "",
        "chapter_terbaru": chapter,
        "rating": rating[-1] if rating else "",
    }
# 202431052_Anfal Ridho Hanif

def valid(d):
    """Layak jika lengkap dan rating > 0 (0.0 = belum dinilai)."""
    try:
        return bool(d["judul"] and d["tipe"] and d["status"]
                    and d["chapter_terbaru"] and float(d["rating"]) > 0)
    except ValueError:
        return False

In [5]:
# 202431052_Anfal Ridho Hanif
result = []
halaman = 1
while halaman <= MAX_PAGE:
    url = LIST_URL.format(halaman)
    r = requests.get(url, headers=HEADERS, timeout=20)         
    print(f"[Halaman {halaman}] status_code = {r.status_code}")
    if r.status_code != 200:
        print("Status bukan 200, berhenti.")
        break

    soup = BeautifulSoup(r.text, "html.parser")                 
    blok = cari_blok(soup)                                      
    print(f"[Halaman {halaman}] jumlah blok data = {len(blok)}")
    if not blok:
        break

    for b in blok:                                              
        result.append(ambil_data(b))

    if sum(valid(d) for d in result) >= TARGET:
        break
    halaman += 1
    time.sleep(DELAY)

print("\nTotal blok terkumpul:", len(result))

[Halaman 1] status_code = 200
[Halaman 1] jumlah blok data = 24
[Halaman 2] status_code = 200
[Halaman 2] jumlah blok data = 24
[Halaman 3] status_code = 200
[Halaman 3] jumlah blok data = 24
[Halaman 4] status_code = 200
[Halaman 4] jumlah blok data = 24
[Halaman 5] status_code = 200
[Halaman 5] jumlah blok data = 24
[Halaman 6] status_code = 200
[Halaman 6] jumlah blok data = 24
[Halaman 7] status_code = 200
[Halaman 7] jumlah blok data = 24
[Halaman 8] status_code = 200
[Halaman 8] jumlah blok data = 24
[Halaman 9] status_code = 200
[Halaman 9] jumlah blok data = 24
[Halaman 10] status_code = 200
[Halaman 10] jumlah blok data = 24
[Halaman 11] status_code = 200
[Halaman 11] jumlah blok data = 24
[Halaman 12] status_code = 200
[Halaman 12] jumlah blok data = 24
[Halaman 13] status_code = 200
[Halaman 13] jumlah blok data = 24
[Halaman 14] status_code = 200
[Halaman 14] jumlah blok data = 24
[Halaman 15] status_code = 200
[Halaman 15] jumlah blok data = 24
[Halaman 16] status_code = 2

In [6]:
# 202431052_Anfal Ridho Hanif
for i, d in enumerate(result, start=1):
    print(i, d["judul"], "|", d["tipe"], "|", d["status"],
          "|", d["chapter_terbaru"], "|", d["rating"])

1 Reincarnated as the Archduke’s Son | Manhwa | Ongoing | 56.0 | 9.7
2 Surviving as the Villainous Younger Sister | Manhwa | Ongoing | 18.0 | 0.0
3 Maou-sama, Retry! R | Manga | Ongoing | 59.2 | 0.0
4 Troll Trap | Manhwa | Hiatus | 262.0 | 0.0
5 The Chairman of Pharmaceutical Company is a Poison King | Manhua | Ongoing | 33.0 | 0.0
6 Temporarily Fostering a Demon | Manhua | Ongoing | 28.0 | 0.0
7 How to Survive as the Villain’s Younger Sibling | Manhwa | Ongoing | 18.0 | 0.0
8 The Supreme Demon Swordmaster | Manhua | Ongoing | 60.0 | 10.0
9 Black Killer Whale Baby | Manhwa | Ongoing | 128.0 | 10.0
10 Student Jo-Eun | Manhwa | Ongoing | 95.0 | 10.0
11 The Saint Levels Up Through Necromancy | Manhwa | Ongoing | 45.0 | 9.0
12 I Received a Request from the Villain Boss | Manhwa | Ongoing | 39.0 | 0.0
13 Futago Matomete Kanojo Ni Shinai | Manga | Ongoing | 20.5 | 10.0
14 Demon Life Starting From A Baby | Manhua | Ongoing | 14.0 | 10.0
15 A Mercenary’s Rebirth Among Nobles | Manhwa | Ongoing

152 Living as a Villainess, Starting With 0 Gold | Manhwa | Ongoing | 21.0 | 10.0
153 Izure Saikyou no Renkinjutsu Shi? | Manga | Ongoing | 60.0 | 10.0
154 The Demon King Overrun by Heroes | Manhwa | Ongoing | 55.0 | 10.0
155 Tomodachi to Koibito, Docchi ga Daiji nano? | Manga | Ongoing | 8.0 | 10.0
156 High-Martial World: One Hand to Overwhelm Three Thousand Emperors! | Manhua | Ongoing | 82.0 | 10.0
157 Please Go Home, Akutsu-san! | Manga | Ongoing | 1275.0 | 0.0
158 Scoring the Sacred Body of the Ancients from the Get-go | Manhua | Ongoing | 176.0 | 10.0
159 A Legend of The Wind | Manhua | Ongoing | 105.2 | 0.0
160 Real Account 2 | Manga | Ongoing | 5253.0 | 0.0
161 Magic Emperor | Manhua | Ongoing | 931.0 | 9.6
162 Bug Eater | Manhwa | Ongoing | 99.0 | 0.0
163 Chairman Kang, The New Employee | Manhwa | Ongoing | 97.0 | 0.0
164 Jiuyang Shenwang | Manhua | Ongoing | 189.0 | 0.0
165 The Regressor’s Tale of Cultivation | Manhwa | Ongoing | 37.0 | 10.0
166 Hatsukoi Algorithm | Manga | O

306 Arifureta Gakuen de Sekai Saikyou | Manga | Ongoing | 24.0 | 0.0
307 Riches Can’t Buy Loyalty | Manhua | Ongoing | 128.0 | 10.0
308 Return of the First Patriarch: The Strongest Reincarnates into His Descendant 1000 Years Later | Manga | Ongoing | 64.0 | 9.3
309 The Little Sprout Who Bloomed in the Villainous Duke's Family | Manhwa | Ongoing | 28.0 | 0.0
310 The Villainous Duke’s Daughter Hides Her Identity | Manhwa | Ongoing | 26.0 | 8.0
311 Pick Me Up | Manhwa | Ongoing | 226.0 | 9.9
312 The Academy’s Weapon Replicator | Manhwa | Ongoing | 25.0 | 9.3
313 Solo Leveling: Ragnarok | Manhwa | Ongoing | 71.0 | 10.0
314 Arcane Sniper | Manhwa | Ongoing | 942.0 | 0.0
315 Ming Dynasty’s Failure | Manhua | Ongoing | 114.0 | 10.0
316 Everyone Else Became a Romance Fantasy Heroine, But I Became a Closet Monster | Manhwa | Ongoing | 23.0 | 0.0
317 Revenge of Reborn Villainess | Manhwa | Ongoing | 19.0 | 0.0
318 Zenmetsu End o Shi ni Monogurui de Kaihi Shita. Party ga Yanda. | Manga | Ongoing 

In [7]:
# 202431052_Anfal Ridho Hanif
kolom = ["judul", "tipe", "status", "chapter_terbaru", "rating"]
with open("komikcast_raw.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=kolom)
    writer.writeheader()
    for d in result:
        writer.writerow(d)
print("Tersimpan: komikcast_raw.csv")

Tersimpan: komikcast_raw.csv


In [8]:
# 202431052_Anfal Ridho Hanif
df = pd.read_csv("komikcast_raw.csv")
display(df)
print(df.dtypes)

,judul,tipe,status,chapter_terbaru,rating
0,Reincarnated as the Archduke’s Son,Manhwa,Ongoing,56.0,9.7
1,Surviving as the Villainous Younger Sister,Manhwa,Ongoing,18.0,0.0
2,"Maou-sama, Retry! R",Manga,Ongoing,59.2,0.0
3,Troll Trap,Manhwa,Hiatus,262.0,0.0
4,The Chairman of Pharmaceutical Company is a Po...,Manhua,Ongoing,33.0,0.0
...,...,...,...,...,...
403,Rebirth: Monarch of the Dead,Manhua,Ongoing,41.0,9.9
404,The Man Who Vowed Revenge on His Demon Wife,Manga,Ongoing,15.0,9.0
405,The Villain Of Destiny,Manhua,Ongoing,368.0,10.0
406,Off Work Today Too,Manhwa,Ongoing,64.0,10.0


judul               object
tipe                object
status              object
chapter_terbaru    float64
rating             float64
dtype: object


In [9]:
# 202431052_Anfal Ridho Hanif
df["chapter_terbaru"] = pd.to_numeric(df["chapter_terbaru"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")


df = df.drop_duplicates(subset="judul")
df = df.dropna(subset=kolom)
df = df[df["rating"] > 0].reset_index(drop=True)

print("Tipe data setelah konversi:")
print(df.dtypes)
print("Jumlah data bersih:", len(df))
assert len(df) >= MIN_DATA, f"Data baru {len(df)}, kurang dari {MIN_DATA}. Naikkan MAX_PAGE."

df.to_csv("komikcast_data.csv", index=False, encoding="utf-8")
print("Tersimpan: komikcast_data.csv")

Tipe data setelah konversi:
judul               object
tipe                object
status              object
chapter_terbaru    float64
rating             float64
dtype: object
Jumlah data bersih: 266
Tersimpan: komikcast_data.csv


In [10]:
# 202431052_Anfal Ridho Hanif
display(df.describe())
print(df["tipe"].value_counts())
print(df["status"].value_counts())

,chapter_terbaru,rating
count,266.000000,266.000000
mean,172.768421,9.617669
std,519.536015,0.859941
min,2.300000,2.000000
25%,33.000000,9.600000
50%,70.000000,10.000000
75%,179.750000,10.000000
max,7199.000000,10.000000


Manhwa    138
Manga      78
Manhua     50
Name: tipe, dtype: int64
Ongoing    262
End          3
Hiatus       1
Name: status, dtype: int64
